# M3.S3 — Profiling and Performance Analysis
## From “it is slow” to an evidence-based diagnosis

This notebook accompanies **M3.S3 — Profiling and Performance Analysis**.

The objective is to move from:

> **“This code looks slow.”**

to:

> **“I measured it, located where time goes, formed a hypothesis, changed one thing, and proved whether it helped.”**

### Classroom method

> **PREDICT → MEASURE → LOCATE → EXPLAIN → CHANGE → REMEASURE**

### What we will do for real

- establish a fair measurement protocol;
- repeat wall-clock timings and inspect variability;
- profile one program with identifiable phases;
- compare **sampling** and **instrumentation**;
- distinguish a **hotspot** from a **bottleneck**;
- collect hardware-counter evidence when the cluster permits it;
- compare poor-locality and good-locality versions of the **same computation**;
- observe profiler overhead;
- measure MPI imbalance as real barrier waiting;
- inspect three evidence excerpts and decide the **next question**, not jump straight to a fix;
- finish with the workflow expected in **Practice 3**.

### Build

**M3S3-2026-09-26-v9**


# 0 — Environment and demo files

### Predict

Before running:

- Is the Jupyter/login node a good place to benchmark?
- Which tools should we use to answer different performance questions?
- Why should the real measurements run inside a declared Slurm allocation?

For this session the live notebook uses the tools we have confirmed on SciTech:

> **GNU time + gprof + strace + OpenMP timing**


In [ ]:
import os
import platform
import re
import shutil
import statistics
import subprocess
import time
import urllib.request
from pathlib import Path

def show_rows(headers, rows):
    widths = [len(str(h)) for h in headers]
    text_rows = [[str(v) for v in row] for row in rows]
    for row in text_rows:
        widths = [max(w, len(v)) for w, v in zip(widths, row)]
    print(" | ".join(str(h).ljust(w) for h, w in zip(headers, widths)))
    print("-+-".join("-" * w for w in widths))
    for row in text_rows:
        print(" | ".join(v.ljust(w) for v, w in zip(row, widths)))

def run_command(command):
    print("$", command)
    p = subprocess.run(command, shell=True, executable="/bin/bash",
                       capture_output=True, text=True)
    if p.stdout.strip():
        print(p.stdout.strip())
    if p.stderr.strip():
        print("[stderr]")
        print(p.stderr.strip())
    print("return code:", p.returncode)
    print()
    return p

print("Host:", platform.node())
print("User:", os.environ.get("USER", "unknown"))
print("Visible CPUs:", os.cpu_count())
print("sbatch:", shutil.which("sbatch"))
print("gcc:", shutil.which("gcc"))
print("gprof:", shutil.which("gprof"))
print("strace:", shutil.which("strace"))
print("GNU time:", "/usr/bin/time" if Path("/usr/bin/time").exists() else None)
print()

DEMO_NAMES = [
    "m3s3_profile_demo.c",
    "m3s3_syscall_demo.c",
    "m3s3_openmp_wait.c",
    "m3s3_profile_lab.sbatch",
]

RAW_BASE = (
    "https://raw.githubusercontent.com/"
    "OscarDiez/hpc_course/main/session_demos/13_profiling_analysis/"
)

FILES_READY = True
for name in DEMO_NAMES:
    try:
        urllib.request.urlretrieve(RAW_BASE + name, name)
        print(f"{name:<28} <- GitHub")
    except Exception as exc:
        FILES_READY = False
        print(f"{name:<28} !! {exc}")

print("\nDemo files ready:", FILES_READY)


### Explain

Seeing many CPUs in Jupyter does **not** mean they are allocated to our benchmark.

For valid performance evidence we want:

> **same workload + declared resources + controlled measurement + repeated runs**

The notebook therefore uses Jupyter to **submit and analyse**, while the real measurements run through Slurm on compute resources.


# 1 — A valid measurement protocol

Before choosing a profiler, define the experiment.

For every comparison, record:

- **workload** — same input and problem size;
- **build** — same code version and compiler flags;
- **resources** — same node type, cores, ranks, GPUs;
- **measurement boundary** — what starts and stops the timer;
- **repetitions** — enough runs to see variability;
- **reported statistic** — median/mean plus variability;
- **one changed variable** — otherwise the comparison is ambiguous.

### Question

Which of these comparisons is fair?

**A** — same input, same node, same binary, before/after one loop-order change  
**B** — different input sizes and different core counts  
**C** — debug build before, `-O3` build after  
**D** — one run before and five runs after

<details>
<summary><strong>Reveal</strong></summary>

**A** is the clean comparison.

</details>


# 2 — The real profile-demo program

The program deliberately has identifiable phases:

```c
initialise(...)
compute_bad(...)   // or compute_good(...)
reduction_work(...)
other_work(...)
```

The key comparison uses the **same matrix** and the **same arithmetic**.

## Bad access order

```c
for (int j = 0; j < n; ++j)
    for (int i = 0; i < n; ++i)
        sum += A[i*n + j];
```

## Good access order

```c
for (int i = 0; i < n; ++i)
    for (int j = 0; j < n; ++j)
        sum += A[i*n + j];
```

Both compute the same sum.

### Predict

1. Which version should run faster in C?
2. Will a profiler tell us **where** time goes?
3. Will that automatically tell us **why**?


# 3 — Run the real profiling lab on the cluster

The live lab now uses two layers of real tools.

## Tools already available on SciTech

- **GNU `time -v`** — whole-program resource analysis;
- **gprof** — real function-level profiling;
- **strace -c** — system-call summary;
- **strace -tt -T** — event-level tracing;
- **OpenMP timing** — per-thread compute and barrier-wait evidence.

## HPC tools loaded through EESSI

The Slurm job initializes the EESSI 2023.06 software layer and looks for:

- **Score-P 8.4** — HPC profiling + event tracing;
- **CubeLib 4.8.2** — command-line inspection of Score-P `.cubex` profiles;
- **Extrae 4.2.0** — another HPC tracing tool available in EESSI.

The main HPC live examples use **Score-P** because one tool can show both profiling and tracing.

The workflow is:

> **measure → locate → profile/trace → form a hypothesis → change one thing → remeasure**


In [ ]:
def clean_slurm_env():
    env = os.environ.copy()
    for key in ("SLURM_MEM_PER_CPU", "SLURM_MEM_PER_GPU", "SLURM_MEM_PER_NODE"):
        env.pop(key, None)
    return env

def submit_slurm(script):
    if shutil.which("sbatch") is None:
        print("sbatch is not available here — run this on SciTech JupyterHub.")
        return None
    p = subprocess.run(
        ["sbatch", "--parsable", script],
        capture_output=True,
        text=True,
        env=clean_slurm_env(),
    )
    if p.returncode != 0:
        print("Slurm submission failed:")
        print(p.stderr)
        return None
    job_id = p.stdout.strip().split(";")[0]
    print("Submitted Slurm job:", job_id)
    return job_id

def wait_for_job(job_id, timeout=900, poll=5):
    if not job_id or shutil.which("squeue") is None:
        return False
    start = time.time()
    while time.time() - start < timeout:
        p = subprocess.run(
            ["squeue", "-h", "-j", str(job_id), "-o", "%T"],
            capture_output=True, text=True
        )
        state = p.stdout.strip()
        if not state:
            print("Job", job_id, "has left the queue.")
            return True
        print("Job", job_id, "state:", state)
        time.sleep(poll)
    print("Timed out waiting. The job may still be queued/running.")
    return False

def read_output(job_id):
    if not job_id:
        return ""
    path = Path(f"m3s3_profile-{job_id}.out")
    if not path.exists():
        return ""
    return path.read_text(errors="replace")

def wait_for_final_output(job_id, timeout=45, poll=2):
    """Wait for the Slurm output file to finish flushing after the job leaves squeue."""
    start = time.time()
    last = ""
    stable_polls = 0

    while time.time() - start < timeout:
        current = read_output(job_id)

        if "M3S3_PROFILE_LAB_RESULT=PASS" in current:
            return current

        if current and current == last:
            stable_polls += 1
        else:
            stable_polls = 0
            last = current

        # If the file has stopped changing for several polls, return it.
        # This also handles jobs that failed before writing the PASS marker.
        if current and stable_polls >= 4:
            return current

        time.sleep(poll)

    return read_output(job_id)

def refresh_output():
    """Reload the output file so later notebook cells never use a stale snapshot."""
    global M3S3_OUTPUT
    if M3S3_JOB_ID:
        latest = read_output(M3S3_JOB_ID)
        if len(latest) >= len(M3S3_OUTPUT):
            M3S3_OUTPUT = latest
    return M3S3_OUTPUT

def section(text, label):
    # Refresh automatically if a later section asks for output after the Slurm file
    # has grown on the shared filesystem.
    if M3S3_JOB_ID:
        text = refresh_output()

    start = f"===== {label} ====="
    end = f"===== END {label} ====="
    i = text.find(start)
    j = text.find(end)
    if i < 0 or j < 0:
        return ""
    return text[i:j + len(end)]

def kv_lines(text, prefix):
    if M3S3_JOB_ID:
        text = refresh_output()

    rows = []
    for line in text.splitlines():
        if not line.startswith(prefix):
            continue
        row = {}
        for token in line.split():
            if "=" in token:
                k, v = token.split("=", 1)
                row[k] = v
        rows.append(row)
    return rows

def show_output_tail(text, lines=35):
    tail = text.splitlines()[-lines:]
    print("\n".join(tail))

print("Slurm helpers ready.")


In [ ]:
# Always refresh the lab files here as well as in Section 0.
# This prevents an old local batch script from being submitted after the
# notebook itself has been updated.
print("Refreshing lab files from GitHub...")
for name in DEMO_NAMES:
    try:
        urllib.request.urlretrieve(RAW_BASE + name, name)
        print(f"  {name}")
    except Exception as exc:
        print(f"  FAILED: {name}: {exc}")

M3S3_JOB_ID = None
M3S3_OUTPUT = ""

if FILES_READY and shutil.which("sbatch"):
    M3S3_JOB_ID = submit_slurm("m3s3_profile_lab.sbatch")
    if wait_for_job(M3S3_JOB_ID):
        M3S3_OUTPUT = wait_for_final_output(M3S3_JOB_ID)

        pass_found = "M3S3_PROFILE_LAB_RESULT=PASS" in M3S3_OUTPUT
        print("\nM3S3 profiling lab finished.")
        print("PASS marker found:", pass_found)

        env_text = section(M3S3_OUTPUT, "ENVIRONMENT")
        if env_text:
            print("\n" + env_text)

        if not pass_found:
            print("\nThe Slurm job finished, but the PASS marker is missing.")
            print("This may mean the job stopped early. Latest output tail:")
            print("-" * 70)
            show_output_tail(M3S3_OUTPUT)
            print("-" * 70)
            print("Later cells will automatically refresh this output file.")
else:
    print("Run this notebook on SciTech JupyterHub to execute the real experiments.")


# 4 — Can you trust one timing?

The job ran the poor-locality version **five times**.

### Predict

- Will all five runs be identical?
- If one run is noticeably slower, should we report only that run?
- Would median be more robust than one arbitrary measurement?


In [ ]:
M3S3_OUTPUT = refresh_output()
bad_totals = [float(r["seconds"]) for r in kv_lines(M3S3_OUTPUT, "PROFILE_TOTAL")
              if r.get("mode") == "bad"]

if bad_totals:
    rows = [(i+1, f"{t:.6f}") for i, t in enumerate(bad_totals[:5])]
    show_rows(["Run", "Wall time (s)"], rows)
    vals = bad_totals[:5]
    print()
    print(f"Median: {statistics.median(vals):.6f} s")
    print(f"Mean:   {statistics.mean(vals):.6f} s")
    print(f"Min:    {min(vals):.6f} s")
    print(f"Max:    {max(vals):.6f} s")
    print(f"Range:  {max(vals)-min(vals):.6f} s")
else:
    print("Run Section 3 first.")


### Explain

One timing is weak evidence.

A reasonable classroom protocol is:

> repeat → inspect variability → report a representative value

The exact statistic depends on the experiment, but the important habit is to make variability visible rather than hide it.


# 5 — Whole-program timing vs region timing

The program prints **phase timings** as well as total time.

That lets us ask two different questions:

- **How long does the whole application take?**
- **Which region contributes most to runtime?**

Run the next cell.


In [ ]:
M3S3_OUTPUT = refresh_output()
rows = [r for r in kv_lines(M3S3_OUTPUT, "PROFILE_PHASE")
        if r.get("mode") == "bad"]

if rows:
    by_phase = {}
    for r in rows:
        by_phase.setdefault(r["phase"], []).append(float(r["seconds"]))
    table = []
    for phase, vals in by_phase.items():
        table.append((phase, f"{statistics.median(vals):.6f}"))
    table.sort(key=lambda x: float(x[1]), reverse=True)
    show_rows(["Phase", "Median region time (s)"], table)
else:
    print("Run Section 3 first.")


### Question

If `compute` dominates total runtime, what do we know?

We know **where a lot of time is spent**.

We do **not yet know why** it is slow.

> **Hotspot = location. Bottleneck = explanation of what limits progress.**


# 6 — Real performance analysis with GNU time

Before a detailed profiler, a simple real tool can already answer useful questions:

```bash
/usr/bin/time -v ./m3s3_profile_demo bad
```

It reports whole-program evidence such as:

- elapsed wall time;
- user/system CPU time;
- maximum resident set size;
- page faults;
- context switches;
- file-system activity.

### Question

Which of these metrics tells us **where inside the code** time is spent?

Answer: none. `time -v` characterizes the process as a whole; it does not locate the hotspot.


In [ ]:
M3S3_OUTPUT = refresh_output()
time_text = section(M3S3_OUTPUT, "GNU TIME RESOURCE ANALYSIS")
print(time_text if time_text else "Run Section 3 first.")


# 7 — Real profiling with gprof: find the hotspot

The Slurm job compiles a second binary with:

```bash
gcc -O3 -g -pg m3s3_profile_demo.c -o m3s3_profile_demo_pg
./m3s3_profile_demo_pg bad 4096 4
gprof ./m3s3_profile_demo_pg gmon.out
```

This is **real profiler output from the compute node**.

### Predict

Which function do you expect to dominate the profile?


In [ ]:
M3S3_OUTPUT = refresh_output()
gprof_text = section(M3S3_OUTPUT, "GPROF INSTRUMENTATION")
print(gprof_text if gprof_text else "Run Section 3 first.")


### Observe

Look for the function with the largest percentage of sampled time.

### What do we know now?

- which function dominates;
- approximately how the runtime is distributed.

### What do we still NOT know?

- whether the hotspot is compute-bound;
- memory-bound;
- branch-limited;
- synchronization-limited;
- or already close to optimal.

A hotspot is a **place to investigate**, not an automatic instruction to rewrite code.


# 8 — Real tracing with strace: summarize system calls

`strace` observes interactions between the process and the Linux kernel.

The teaching program performs 2,000 small writes:

```bash
strace -c ./m3s3_syscall_demo
```

The summary reports:

- syscall name;
- number of calls;
- total time;
- time per call;
- errors.

### Predict

Which syscall should dominate the call count?


In [ ]:
M3S3_OUTPUT = refresh_output()
trace_text = section(M3S3_OUTPUT, "STRACE SUMMARY")
print(trace_text if trace_text else "Run Section 3 first.")


### Explain

This is **tracing**, not profiling.

The tool records concrete operating-system events.

For HPC applications, traces are useful when the question concerns:

- system calls;
- file access;
- synchronization;
- communication events;
- timing relationships between activities.

A trace can become very large, so we should trace only when the question requires event-level detail.


# 9 — Real event trace: see operations in time order

A summary tells us **how much** happened. A trace shows **what happened and when**.

The same program is also run with:

```bash
strace -tt -T -e trace=openat,write,close,unlink ./m3s3_syscall_demo
```

- `-tt` adds timestamps;
- `-T` adds syscall duration;
- `-e trace=...` keeps the trace focused.

### Observe

Look at the first events. Can you identify the repeated `write()` calls?


In [ ]:
M3S3_OUTPUT = refresh_output()
event_trace = section(M3S3_OUTPUT, "STRACE EVENT TRACE")
print(event_trace if event_trace else "Run Section 3 first.")


### Explain

This is genuine **event-level tracing**.

A trace is useful when the performance question involves:

- file or network activity;
- synchronization;
- communication;
- repeated small operations;
- timing relationships between events.

The trade-off is volume: detailed traces can become very large.


# 10 — Sampling vs instrumentation

## Sampling

Periodically asks:

> **Where is the program executing right now?**

A common Linux tool is:

```bash
perf record -F 99 -g ./app
perf report
```

Sampling is often useful for finding hotspots with relatively low overhead.

## Instrumentation

Adds measurement hooks so execution is explicitly recorded.

Our real example is:

```bash
gcc -pg ...
./app
gprof ...
```

### SciTech note

`perf` is **not installed on the current SciTech compute image**, so we do not pretend to run it here. We use `gprof`, `strace`, explicit timers and OpenMP timing for the live class.

> **Choose the least intrusive tool that can answer the performance question.**


# 11 — Load real HPC profiling tools through EESSI

The cluster already exposes the EESSI filesystem through CVMFS.

Inside the **Slurm compute job** we initialize the full EESSI module environment:

```bash
source /cvmfs/software.eessi.io/versions/2023.06/init/lmod/bash
```

and then inspect/load:

```bash
module avail Score-P
module avail Extrae
module avail CubeLib
```

For this lesson the batch job attempts:

```bash
module load Score-P/8.4-gompi-2023b
module load CubeLib/4.8.2-GCCcore-13.2.0
```

### Why this is useful

We do **not** install separate copies for every student.

EESSI provides the same HPC tools from a centrally managed software stack.


In [ ]:
M3S3_OUTPUT = refresh_output()
eessi = section(M3S3_OUTPUT, "EESSI PROFILING TOOLS")
print(eessi if eessi else "Run Section 3 first.")


### What to look for

A successful discovery should show a **Score-P 8.4** module and an **Extrae 4.2.0** module.

Score-P becomes our main HPC tool because it can produce:

- a **profile**: aggregated time/call-path information;
- a **trace**: event information across threads/processes.

That makes the distinction between profiling and tracing concrete.


# 12 — Real Score-P profile: instrument a C application

Score-P works as a compiler wrapper.

The job builds the same poor-locality program through Score-P:

```bash
scorep gcc -O2 -g m3s3_profile_demo.c -o m3s3_profile_demo_scorep
```

and runs:

```bash
SCOREP_EXPERIMENT_DIRECTORY=scorep_serial_bad \
    ./m3s3_profile_demo_scorep bad 2048 2
```

The result is a real Score-P experiment directory containing:

```text
profile.cubex
```

The batch job then uses:

```bash
scorep-score profile.cubex
cube_dump -m time ... profile.cubex
```

to inspect the profile from the command line.

### Predict

Which region/function should dominate the poor-locality run?


In [ ]:
M3S3_OUTPUT = refresh_output()
scorep_serial = section(M3S3_OUTPUT, "SCOREP SERIAL PROFILE")
print(scorep_serial if scorep_serial else "Run Section 3 first.")


### Explain

This is now a genuine **HPC performance profile**.

Compare it with `gprof`:

- both can help locate where time goes;
- Score-P is designed for parallel HPC applications and integrates MPI/OpenMP measurement;
- the `.cubex` profile can be consumed by Cube/Scalasca tooling.

The profiler tells us **where** the application spends time. We still need a hypothesis for **why**.


# 13 — Real Score-P OpenMP profile: see parallel regions

Now the same HPC tool instruments the four-thread imbalance example:

```bash
scorep gcc -O2 -g -fopenmp m3s3_openmp_wait.c \
    -o m3s3_openmp_wait_scorep
```

It runs with:

```bash
OMP_NUM_THREADS=4
SCOREP_EXPERIMENT_DIRECTORY=scorep_openmp_profile
./m3s3_openmp_wait_scorep
```

The program deliberately gives thread 3 more work.

### Predict

What should an HPC profiler reveal around the parallel region and synchronization?


In [ ]:
M3S3_OUTPUT = refresh_output()
scorep_omp = section(M3S3_OUTPUT, "SCOREP OPENMP PROFILE")
print(scorep_omp if scorep_omp else "Run Section 3 first.")


### Explain

This is the important progression:

> **serial profile → parallel profile**

The same measurement infrastructure can now observe OpenMP execution.

Our explicit per-thread timing still gives the clearest small classroom table, while Score-P provides the HPC-style profile that scales to much larger applications.


# 14 — Real Score-P trace: profile vs trace

For the same OpenMP program we switch Score-P from profiling to tracing:

```bash
SCOREP_ENABLE_PROFILING=false
SCOREP_ENABLE_TRACING=true
SCOREP_EXPERIMENT_DIRECTORY=scorep_openmp_trace
OMP_NUM_THREADS=4 ./m3s3_openmp_wait_scorep_trace
```

A **profile** aggregates observations.

A **trace** records events so their temporal relationships can be reconstructed.

The job lists the real OTF2/Score-P files created in the experiment directory.

### Predict

Which will generally generate more data: a profile or a trace?


In [ ]:
M3S3_OUTPUT = refresh_output()
scorep_trace = section(M3S3_OUTPUT, "SCOREP OPENMP TRACE")
print(scorep_trace if scorep_trace else "Run Section 3 first.")


### Explain

The trace directory is evidence that Score-P captured event data rather than only an aggregate profile.

Tracing is valuable for questions such as:

- when did each thread enter/leave a region?
- when did synchronization occur?
- which thread/process was waiting?
- how did communication and computation overlap?

But trace volume can grow quickly.

> **Profile first. Trace when the performance question needs event-level timing.**

### Optional extension

EESSI also provides **Extrae**, which can generate traces for post-mortem analysis in Paraver. We keep it as an extension rather than requiring two tracing frameworks in the same 80-minute lesson.


# 15 — Change ONE thing and remeasure

We now compare:

```text
BAD:  column-style traversal of a C row-major array
GOOD: row-style traversal of the same array
```

Same:

- matrix size;
- arithmetic;
- compiler;
- node allocation;
- number of runs.

Only the **access order** changes.

### Predict

If poor locality is the main limitation, what should happen to runtime?


In [ ]:
M3S3_OUTPUT = refresh_output()
bad_vals = [float(r["seconds"]) for r in kv_lines(M3S3_OUTPUT, "PROFILE_TOTAL")
            if r.get("mode") == "bad"][:5]
good_vals = [float(r["seconds"]) for r in kv_lines(M3S3_OUTPUT, "PROFILE_TOTAL")
             if r.get("mode") == "good"][:5]

if bad_vals and good_vals:
    b = statistics.median(bad_vals)
    g = statistics.median(good_vals)
    show_rows(
        ["Version", "Median wall time (s)", "Relative"],
        [
            ("Bad locality", f"{b:.6f}", "1.00x"),
            ("Good locality", f"{g:.6f}", f"{b/g:.2f}x faster"),
        ],
    )
    print()
    print("Same computation result is checked by the program checksum.")
else:
    print("Run Section 3 first.")


### Evidence-led conclusion

If the good-locality version is substantially faster, that supports the hypothesis:

> **memory access order was limiting performance**

The reasoning chain is:

**baseline → hotspot → evidence → hypothesis → one change → remeasure**

That is stronger than:

> “This loop looked inefficient, so I rewrote it.”


# 16 — Measurement has a cost: the observer effect

Profiling changes the program.

The job runs:

- a normal optimized binary;
- a `-pg` instrumented binary.

### Question

Should the instrumented runtime be assumed identical to the baseline runtime?


In [ ]:
M3S3_OUTPUT = refresh_output()
normal = bad_totals[:5] if 'bad_totals' in globals() else []
pg_rows = []
inside = section(M3S3_OUTPUT, "GPROF INSTRUMENTATION")
for line in inside.splitlines():
    if line.startswith("PROFILE_TOTAL"):
        row = {}
        for token in line.split():
            if "=" in token:
                k,v = token.split("=",1)
                row[k]=v
        if row.get("mode") == "bad":
            pg_rows.append(float(row["seconds"]))

if normal:
    print(f"Normal baseline median: {statistics.median(normal):.6f} s")
if pg_rows:
    print(f"Instrumented run:       {pg_rows[0]:.6f} s")
    print(f"Instrumented / baseline: {pg_rows[0]/statistics.median(normal):.3f}x")
else:
    print("The gprof run redirected program stdout in the batch script,")
    print("so use this section conceptually and the profiler output above.")


### Takeaway

More detailed observation can mean:

- more overhead;
- more data;
- altered timing behavior.

> **Start coarse. Add detail when the question requires it.**


# 17 — Parallel profiling: real OpenMP imbalance and barrier waiting

Now we profile a **parallel** problem.

Four OpenMP threads run the same function, but thread 3 receives four times more work:

```c
iters = (tid == 3) ? 4 * base : base;
```

Each thread measures:

```c
compute_time
barrier_wait_time
```

### Predict

Which threads should spend the most time waiting at the barrier?


In [ ]:
M3S3_OUTPUT = refresh_output()
rows = kv_lines(M3S3_OUTPUT, "OMP_TRACE ")
required = {"thread", "work_iters", "compute", "barrier_wait"}
rows = [r for r in rows if required.issubset(r)]

if rows:
    table = []
    for r in sorted(rows, key=lambda x: int(x["thread"])):
        table.append((
            r["thread"],
            r["work_iters"],
            f'{float(r["compute"]):.6f}',
            f'{float(r["barrier_wait"]):.6f}',
        ))
    show_rows(["Thread", "Work iterations", "Compute (s)", "Barrier wait (s)"], table)
else:
    print("OpenMP trace data unavailable. Run Section 3 first.")


### Explain

The barrier is where waiting is **observed**.

But the barrier is not necessarily the root cause.

If thread 3 does much more work, the likely cause is:

> **load imbalance**

This is an important profiling habit:

> do not optimize the place where a symptom appears before identifying the cause.


# 18 — Performance detective: three evidence excerpts

Work in pairs.

For each excerpt, answer:

1. What do we know?
2. What do we **not** know?
3. What is the **next question or measurement**?

Do **not** jump directly to a code change.


In [ ]:
M3S3_OUTPUT = refresh_output()

print("EXCERPT A — HOTSPOT")
rows = [r for r in kv_lines(M3S3_OUTPUT, "PROFILE_PHASE") if r.get("mode") == "bad"]
if rows:
    by = {}
    for r in rows:
        by.setdefault(r["phase"], []).append(float(r["seconds"]))
    total = sum(statistics.median(v) for v in by.values())
    table = []
    for phase, vals in sorted(by.items(), key=lambda kv: statistics.median(kv[1]), reverse=True):
        med = statistics.median(vals)
        table.append((phase, f"{100*med/total:.1f}%"))
    show_rows(["Phase", "Approx. share"], table)

print("\nEXCERPT B — SYSTEM-CALL TRACE")
trace = section(M3S3_OUTPUT, "STRACE SUMMARY")
for line in trace.splitlines():
    if "write" in line or "total" in line:
        print(line)

print("\nEXCERPT C — PARALLEL WAITING")
rows = kv_lines(M3S3_OUTPUT, "OMP_TRACE ")
required = {"thread", "compute", "barrier_wait"}
rows = [r for r in rows if required.issubset(r)]
if rows:
    table = [
        (r["thread"], f'{float(r["compute"]):.4f}', f'{float(r["barrier_wait"]):.4f}')
        for r in sorted(rows, key=lambda x: int(x["thread"]))
    ]
    show_rows(["Thread", "Compute(s)", "Barrier wait(s)"], table)


### Suggested reasoning

**A — Hotspot**

Next question:

> Why is the dominant compute phase slow?

Possible evidence: access pattern, algorithmic work, compiler behaviour.

**B — Trace**

If `write()` dominates the syscall count:

> Are we performing too many small I/O operations?

A next experiment could batch many small writes into fewer large writes.

**C — Parallel waiting**

If thread 3 computes longer while threads 0–2 wait:

> Why does thread 3 have more work?

Investigate work distribution before trying to remove or tune the barrier.

### Reveal after discussion

The learning outcome is not “choose the fix immediately.”

It is:

> **choose the next measurement that reduces uncertainty.**


# 19 — Premature vs evidence-led optimisation

## Premature optimisation

```text
“This loop looks slow.”
        ↓
rewrite code
        ↓
hope
```

## Evidence-led optimisation

```text
Question
  ↓
Baseline
  ↓
Profile
  ↓
Hypothesis
  ↓
Change ONE thing
  ↓
Remeasure
  ↓
Keep / reject the change
```

### Question

If an optimisation makes one function 2× faster but total application runtime changes by only 1%, was it the right target?


# 20 — Practice 3 preview: prove an improvement

Your mission in Practice 3 is not simply:

> **make code faster**

It is:

> **prove that you identified a bottleneck and improved it using valid performance evidence**

Minimum evidence chain:

1. define the performance question;
2. establish a reproducible baseline;
3. locate a hotspot or symptom;
4. collect evidence relevant to the suspected bottleneck;
5. state an optimisation hypothesis;
6. change one thing;
7. rerun under comparable conditions;
8. report improvement **and limitations**.

### P3 pre-flight

Before the practice, make sure you can produce:

- one clean baseline run;
- the exact build command;
- the exact Slurm resource request;
- the input/problem size;
- enough repetitions to judge variability.


# 21 — Exit ticket

Answer briefly.

1. Why is one timing weak evidence?
2. What is the difference between a hotspot and a bottleneck?
3. When would you prefer sampling over heavy instrumentation?
4. What does a hardware counter provide: proof or a clue?
5. Why should we change one variable at a time?
6. If many MPI ranks wait at a barrier, where should we investigate first?
7. What is the final step after an optimisation?

<details>
<summary><strong>Concise answers</strong></summary>

1. Performance varies; one run does not reveal variability.
2. A hotspot is where time is spent; a bottleneck explains what limits progress.
3. When we need low-overhead localisation of CPU time/hotspots.
4. A clue that must be interpreted with other evidence.
5. So we can attribute the performance difference to that change.
6. The work before the barrier, especially the slow/critical rank.
7. Remeasure under comparable conditions and decide whether the evidence supports keeping the change.

</details>


# What you should leave with

- **Measure before profiling.**
- Define the performance question before choosing the tool.
- Repeated measurements are stronger evidence than one run.
- Sampling helps locate **where** CPU time goes.
- Instrumentation can add detail, but may add overhead.
- A **hotspot is not automatically a bottleneck**.
- Hardware counters provide clues about what the CPU experiences.
- Tracing/wait measurements help explain parallel behavior over time.
- Build a hypothesis from multiple pieces of evidence.
- Change **one thing**.
- Remeasure under comparable conditions.
- Keep an optimisation only when the evidence supports it.

The workflow is:

> **MEASURE → LOCATE → EXPLAIN → CHANGE → REMEASURE**

Next: **M3.S4 — Libraries, I/O and Reproducible Workflows**.
